# R1-A3-S7 Redes Neuronales III
**Deep Learning - Conceptos** · Especialización en Inteligencia Artificial
**Autor:** Juan Camilo Cruz Beltran

Este notebook implementa, con **TensorFlow + Keras** sobre el dataset **MNIST**:

1. Preprocesamiento y distribución de datos (train / dev / test)
2. Técnicas de optimización (SGD, SGD con momentum, RMSprop, Adam) y efecto del learning rate
3. Métodos de regularización (L1, L2, Dropout, Early Stopping)
4. Análisis de bias, variance y tradeoff
5. Métricas de evaluación (Accuracy, Precision, Recall, F1 y matriz de confusión)
6. Generación automática del informe en PDF con los resultados reales

**Cómo usarlo:** en Colab ve a *Entorno de ejecución > Ejecutar todo*. Antes, en la última celda, cambia la variable `GITHUB_URL` por el enlace de tu repositorio. Al terminar se genera `Informe_Redes_Neuronales_III.pdf`.

## 0. Configuración del entorno
Se importan las librerías y se fija una **semilla** para que los resultados sean reproducibles.

In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

os.makedirs("figuras", exist_ok=True)   # aquí se guardan las gráficas para el informe
os.makedirs("resultados", exist_ok=True)
print("TensorFlow:", tf.__version__)

## 1. Distribución y preprocesamiento de datos
- **Train:** el modelo aprende (ajusta pesos) con estos datos.
- **Dev (validación):** sirve para comparar hiperparámetros y detectar overfitting. No se usa para entrenar.
- **Test:** se usa **una sola vez al final** para estimar el desempeño real. Si se usara para decidir hiperparámetros, la estimación sería optimista.

Preprocesamiento: normalización de píxeles de [0, 255] a [0, 1], aplanado de 28x28 a 784 características y división **estratificada** (mantiene la proporción de cada clase).

In [ ]:
# Carga de MNIST: 60.000 imágenes de entrenamiento y 10.000 de prueba (dígitos 0-9)
(x_full, y_full), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalización a [0, 1]
x_full = x_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Muestras de ejemplo
fig, axes = plt.subplots(2, 5, figsize=(10, 4.2))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(x_full[i], cmap="gray")
    ax.set_title(f"Etiqueta: {y_full[i]}")
    ax.axis("off")
plt.tight_layout()
plt.savefig("figuras/01_muestras_mnist.png", dpi=130)
plt.show()

# Aplanado 28x28 -> 784
x_full = x_full.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

# División train / dev (10 % para dev), estratificada por clase
x_train, x_dev, y_train, y_dev = train_test_split(
    x_full, y_full, test_size=0.10, random_state=SEED, stratify=y_full)

n_train, n_dev, n_test = len(x_train), len(x_dev), len(x_test)
print(f"Train: {x_train.shape} | Dev: {x_dev.shape} | Test: {x_test.shape}")

# Distribución de clases en train
plt.figure(figsize=(7, 3.5))
plt.bar(range(10), np.bincount(y_train), color="#4C78A8")
plt.xticks(range(10))
plt.xlabel("Clase (dígito)")
plt.ylabel("Cantidad de imágenes")
plt.title("Distribución de clases en el conjunto de entrenamiento")
plt.tight_layout()
plt.savefig("figuras/02_distribucion_clases.png", dpi=130)
plt.show()

## 2. Funciones auxiliares
- `build_model`: construye un perceptrón multicapa (MLP) con capas densas ReLU y salida softmax de 10 clases. Recibe los **hiperparámetros** (optimizador, neuronas por capa, dropout, L1, L2).
- `train`: entrena usando el conjunto dev como validación.
- `plot_curves`: grafica curvas por época.

In [ ]:
def build_model(optimizer, units=(256, 128), dropout=0.0, l1=0.0, l2=0.0):
    # Regularizador de pesos (L1, L2 o ambos) aplicado a las capas ocultas
    reg = regularizers.l1_l2(l1=l1, l2=l2) if (l1 > 0 or l2 > 0) else None

    model = keras.Sequential([layers.Input(shape=(784,))])
    for u in units:
        model.add(layers.Dense(u, activation="relu", kernel_regularizer=reg))
        if dropout > 0:
            model.add(layers.Dropout(dropout))   # apaga neuronas al azar solo en entrenamiento
    model.add(layers.Dense(10, activation="softmax"))

    model.compile(optimizer=optimizer,
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model


def train(model, epochs=10, batch_size=128, callbacks=None):
    return model.fit(x_train, y_train,
                     validation_data=(x_dev, y_dev),
                     epochs=epochs, batch_size=batch_size,
                     callbacks=callbacks, verbose=0)


def plot_curves(histories, metric, title, ylabel, filename):
    plt.figure(figsize=(8, 4.5))
    for name, h in histories.items():
        plt.plot(range(1, len(h.history[metric]) + 1), h.history[metric], label=name)
    plt.title(title)
    plt.xlabel("Época")
    plt.ylabel(ylabel)
    plt.grid(alpha=0.3)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(filename, dpi=130)
    plt.show()

## 3. Técnicas de optimización
Se entrena la **misma arquitectura** (784-256-128-10) cambiando solo el optimizador y el learning rate, para aislar su efecto:

- **SGD:** actualiza los pesos con el gradiente de cada mini-batch.
- **SGD + Momentum:** acumula una "velocidad" de gradientes pasados, lo que suaviza y acelera la convergencia.
- **RMSprop:** adapta el learning rate por parámetro según la magnitud reciente de sus gradientes.
- **Adam:** combina momentum y adaptación del learning rate.

El **learning rate** es el hiperparámetro más sensible: muy alto produce inestabilidad, muy bajo hace el aprendizaje lento.

In [ ]:
EPOCHS = 10

opt_configs = {
    "SGD (lr=0.01)":                 lambda: keras.optimizers.SGD(learning_rate=0.01),
    "SGD + Momentum (lr=0.01, m=0.9)": lambda: keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    "RMSprop (lr=0.001)":            lambda: keras.optimizers.RMSprop(learning_rate=0.001),
    "Adam (lr=0.01)":                lambda: keras.optimizers.Adam(learning_rate=0.01),
    "Adam (lr=0.001)":               lambda: keras.optimizers.Adam(learning_rate=0.001),
    "Adam (lr=0.0001)":              lambda: keras.optimizers.Adam(learning_rate=0.0001),
}

opt_histories, opt_rows = {}, []
for name, make_opt in opt_configs.items():
    keras.utils.set_random_seed(SEED)           # misma inicialización para comparar justo
    model = build_model(make_opt())
    h = train(model, EPOCHS)
    opt_histories[name] = h
    opt_rows.append({
        "Optimizador": name,
        "Acc train": h.history["accuracy"][-1],
        "Acc dev": h.history["val_accuracy"][-1],
        "Loss dev": h.history["val_loss"][-1],
    })
    print(f"{name:35s} acc dev = {h.history['val_accuracy'][-1]:.4f}")

opt_df = pd.DataFrame(opt_rows).round(4)
opt_df.to_csv("resultados/optimizacion.csv", index=False)
display(opt_df)

plot_curves(opt_histories, "val_loss", "Pérdida en dev por optimizador", "Pérdida (dev)",
            "figuras/03_opt_val_loss.png")
plot_curves(opt_histories, "val_accuracy", "Accuracy en dev por optimizador", "Accuracy (dev)",
            "figuras/04_opt_val_accuracy.png")

## 4. Regularización, bias y variance
Se usa una red **más grande** (784-512-512-256-10) y más épocas (30) con Adam (lr=0.001), para que el overfitting sea observable. Luego se aplican:

- **L1:** penaliza la suma de |pesos| (tiende a pesos exactamente cero).
- **L2:** penaliza la suma de pesos al cuadrado (pesos pequeños, "weight decay").
- **Dropout:** apaga una fracción de neuronas en cada paso de entrenamiento.
- **Early Stopping:** detiene el entrenamiento cuando la pérdida en dev deja de mejorar y restaura los mejores pesos.

**Bias y variance:** el *error en train* aproxima el **sesgo** (underfitting si es alto); la *brecha* error dev - error train aproxima la **varianza** (overfitting si es grande). El **tradeoff** consiste en reducir la varianza sin aumentar demasiado el sesgo.

In [ ]:
EPOCHS_REG = 30
BIG = (512, 512, 256)

def make_adam():
    return keras.optimizers.Adam(learning_rate=0.001)

def make_early():
    return keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

reg_configs = {
    "Sin regularización":            dict(kw={},                          early=False),
    "L2 (1e-4)":                     dict(kw=dict(l2=1e-4),               early=False),
    "L1 (1e-5)":                     dict(kw=dict(l1=1e-5),               early=False),
    "Dropout (0.3)":                 dict(kw=dict(dropout=0.3),           early=False),
    "Early Stopping":                dict(kw={},                          early=True),
    "L2 + Dropout + Early Stopping": dict(kw=dict(l2=1e-4, dropout=0.3),  early=True),
}

reg_histories, reg_models, reg_rows = {}, {}, []
for name, cfg in reg_configs.items():
    keras.utils.set_random_seed(SEED)
    model = build_model(make_adam(), units=BIG, **cfg["kw"])
    cbs = [make_early()] if cfg["early"] else None
    h = train(model, EPOCHS_REG, callbacks=cbs)

    # Evaluación final con los pesos finales (con Early Stopping, los mejores restaurados)
    _, acc_tr = model.evaluate(x_train, y_train, verbose=0)
    _, acc_dv = model.evaluate(x_dev, y_dev, verbose=0)

    reg_histories[name] = h
    reg_models[name] = model
    reg_rows.append({
        "Configuración": name,
        "Épocas": len(h.history["loss"]),
        "Acc train": acc_tr,
        "Acc dev": acc_dv,
        "Error train (sesgo)": 1 - acc_tr,
        "Error dev": 1 - acc_dv,
        "Brecha (varianza)": acc_tr - acc_dv,
    })
    print(f"{name:32s} acc train = {acc_tr:.4f} | acc dev = {acc_dv:.4f}")

reg_df = pd.DataFrame(reg_rows).round(4)
reg_df.to_csv("resultados/regularizacion.csv", index=False)
display(reg_df)

# Evidencia de overfitting: pérdida train vs dev del modelo sin regularización
h0 = reg_histories["Sin regularización"]
plt.figure(figsize=(8, 4.5))
plt.plot(range(1, len(h0.history["loss"]) + 1), h0.history["loss"], label="Train")
plt.plot(range(1, len(h0.history["val_loss"]) + 1), h0.history["val_loss"], label="Dev")
plt.title("Modelo sin regularización: pérdida train vs dev")
plt.xlabel("Época"); plt.ylabel("Pérdida"); plt.grid(alpha=0.3); plt.legend()
plt.tight_layout()
plt.savefig("figuras/05_overfitting_baseline.png", dpi=130)
plt.show()

plot_curves(reg_histories, "val_accuracy", "Accuracy en dev por técnica de regularización",
            "Accuracy (dev)", "figuras/06_reg_val_accuracy.png")

## 5. Evaluación final con métricas de clasificación
Se elige la configuración con **mejor accuracy en dev** y se evalúa **una sola vez** en test.

- **Accuracy:** proporción de aciertos totales.
- **Precision:** de lo que el modelo predijo como clase *c*, qué fracción era realmente *c*.
- **Recall:** de lo que realmente era clase *c*, qué fracción detectó el modelo.
- **F1:** media armónica entre precision y recall.
- **Matriz de confusión:** filas = clase real, columnas = clase predicha.

Para las métricas globales se usa promedio **macro** (cada clase pesa igual).

In [ ]:
best_name = reg_df.sort_values("Acc dev").iloc[-1]["Configuración"]
best_model = reg_models[best_name]
print("Modelo seleccionado (mejor en dev):", best_name)

y_prob = best_model.predict(x_test, verbose=0)
y_pred = y_prob.argmax(axis=1)

final_metrics = {
    "Accuracy":  accuracy_score(y_test, y_pred),
    "Precision (macro)": precision_score(y_test, y_pred, average="macro"),
    "Recall (macro)":    recall_score(y_test, y_pred, average="macro"),
    "F1 (macro)":        f1_score(y_test, y_pred, average="macro"),
}
for k, v in final_metrics.items():
    print(f"{k:20s}: {v:.4f}")

print()
print(classification_report(y_test, y_pred, digits=4))
report_dict = classification_report(y_test, y_pred, digits=4, output_dict=True)

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=list(range(10))).plot(
    ax=ax, cmap="Blues", values_format="d", colorbar=False)
ax.set_title(f"Matriz de confusión en test - {best_name}")
plt.tight_layout()
plt.savefig("figuras/07_matriz_confusion.png", dpi=130)
plt.show()

# Confusiones más frecuentes (fuera de la diagonal)
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)
idx = np.argsort(cm_off.ravel())[::-1][:3]
confusions = [(int(i // 10), int(i % 10), int(cm_off.ravel()[i])) for i in idx]
for real, pred, n in confusions:
    print(f"Real {real} predicho como {pred}: {n} veces")

pd.DataFrame([final_metrics]).round(4).to_csv("resultados/metricas_test.csv", index=False)

## 6. Generación automática del informe PDF
Esta celda arma el PDF con el código, las tablas, las gráficas y las explicaciones, usando los **resultados reales** de esta ejecución.
**Antes de ejecutarla, cambia `GITHUB_URL` por el enlace de tu repositorio.**

In [ ]:
!pip -q install reportlab

import textwrap
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.lib.enums import TA_JUSTIFY, TA_CENTER
from reportlab.lib.utils import ImageReader
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, PageBreak,
                                Image, Table, TableStyle, Preformatted, KeepTogether)

# ====== EDITA ESTO ======
GITHUB_URL = "https://github.com/TU_USUARIO/TU_REPOSITORIO"
AUTOR = "Juan Camilo Cruz Beltran"
# ========================

CODE = {'setup': 'import os, random\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport tensorflow as tf\nfrom tensorflow import keras\nfrom tensorflow.keras import layers, regularizers\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,\n                             classification_report, confusion_matrix, ConfusionMatrixDisplay)\nfrom IPython.display import display\n\nSEED = 42\nrandom.seed(SEED)\nnp.random.seed(SEED)\ntf.random.set_seed(SEED)\n\nos.makedirs("figuras", exist_ok=True)   # aquí se guardan las gráficas para el informe\nos.makedirs("resultados", exist_ok=True)\nprint("TensorFlow:", tf.__version__)', 'datos': '# Carga de MNIST: 60.000 imágenes de entrenamiento y 10.000 de prueba (dígitos 0-9)\n(x_full, y_full), (x_test, y_test) = keras.datasets.mnist.load_data()\n\n# Normalización a [0, 1]\nx_full = x_full.astype("float32") / 255.0\nx_test = x_test.astype("float32") / 255.0\n\n# Muestras de ejemplo\nfig, axes = plt.subplots(2, 5, figsize=(10, 4.2))\nfor i, ax in enumerate(axes.ravel()):\n    ax.imshow(x_full[i], cmap="gray")\n    ax.set_title(f"Etiqueta: {y_full[i]}")\n    ax.axis("off")\nplt.tight_layout()\nplt.savefig("figuras/01_muestras_mnist.png", dpi=130)\nplt.show()\n\n# Aplanado 28x28 -> 784\nx_full = x_full.reshape(-1, 784)\nx_test = x_test.reshape(-1, 784)\n\n# División train / dev (10 % para dev), estratificada por clase\nx_train, x_dev, y_train, y_dev = train_test_split(\n    x_full, y_full, test_size=0.10, random_state=SEED, stratify=y_full)\n\nn_train, n_dev, n_test = len(x_train), len(x_dev), len(x_test)\nprint(f"Train: {x_train.shape} | Dev: {x_dev.shape} | Test: {x_test.shape}")\n\n# Distribución de clases en train\nplt.figure(figsize=(7, 3.5))\nplt.bar(range(10), np.bincount(y_train), color="#4C78A8")\nplt.xticks(range(10))\nplt.xlabel("Clase (dígito)")\nplt.ylabel("Cantidad de imágenes")\nplt.title("Distribución de clases en el conjunto de entrenamiento")\nplt.tight_layout()\nplt.savefig("figuras/02_distribucion_clases.png", dpi=130)\nplt.show()', 'funciones': 'def build_model(optimizer, units=(256, 128), dropout=0.0, l1=0.0, l2=0.0):\n    # Regularizador de pesos (L1, L2 o ambos) aplicado a las capas ocultas\n    reg = regularizers.l1_l2(l1=l1, l2=l2) if (l1 > 0 or l2 > 0) else None\n\n    model = keras.Sequential([layers.Input(shape=(784,))])\n    for u in units:\n        model.add(layers.Dense(u, activation="relu", kernel_regularizer=reg))\n        if dropout > 0:\n            model.add(layers.Dropout(dropout))   # apaga neuronas al azar solo en entrenamiento\n    model.add(layers.Dense(10, activation="softmax"))\n\n    model.compile(optimizer=optimizer,\n                  loss="sparse_categorical_crossentropy",\n                  metrics=["accuracy"])\n    return model\n\n\ndef train(model, epochs=10, batch_size=128, callbacks=None):\n    return model.fit(x_train, y_train,\n                     validation_data=(x_dev, y_dev),\n                     epochs=epochs, batch_size=batch_size,\n                     callbacks=callbacks, verbose=0)\n\n\ndef plot_curves(histories, metric, title, ylabel, filename):\n    plt.figure(figsize=(8, 4.5))\n    for name, h in histories.items():\n        plt.plot(range(1, len(h.history[metric]) + 1), h.history[metric], label=name)\n    plt.title(title)\n    plt.xlabel("Época")\n    plt.ylabel(ylabel)\n    plt.grid(alpha=0.3)\n    plt.legend(fontsize=8)\n    plt.tight_layout()\n    plt.savefig(filename, dpi=130)\n    plt.show()', 'optimizacion': 'EPOCHS = 10\n\nopt_configs = {\n    "SGD (lr=0.01)":                 lambda: keras.optimizers.SGD(learning_rate=0.01),\n    "SGD + Momentum (lr=0.01, m=0.9)": lambda: keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),\n    "RMSprop (lr=0.001)":            lambda: keras.optimizers.RMSprop(learning_rate=0.001),\n    "Adam (lr=0.01)":                lambda: keras.optimizers.Adam(learning_rate=0.01),\n    "Adam (lr=0.001)":               lambda: keras.optimizers.Adam(learning_rate=0.001),\n    "Adam (lr=0.0001)":              lambda: keras.optimizers.Adam(learning_rate=0.0001),\n}\n\nopt_histories, opt_rows = {}, []\nfor name, make_opt in opt_configs.items():\n    keras.utils.set_random_seed(SEED)           # misma inicialización para comparar justo\n    model = build_model(make_opt())\n    h = train(model, EPOCHS)\n    opt_histories[name] = h\n    opt_rows.append({\n        "Optimizador": name,\n        "Acc train": h.history["accuracy"][-1],\n        "Acc dev": h.history["val_accuracy"][-1],\n        "Loss dev": h.history["val_loss"][-1],\n    })\n    print(f"{name:35s} acc dev = {h.history[\'val_accuracy\'][-1]:.4f}")\n\nopt_df = pd.DataFrame(opt_rows).round(4)\nopt_df.to_csv("resultados/optimizacion.csv", index=False)\ndisplay(opt_df)\n\nplot_curves(opt_histories, "val_loss", "Pérdida en dev por optimizador", "Pérdida (dev)",\n            "figuras/03_opt_val_loss.png")\nplot_curves(opt_histories, "val_accuracy", "Accuracy en dev por optimizador", "Accuracy (dev)",\n            "figuras/04_opt_val_accuracy.png")', 'regularizacion': 'EPOCHS_REG = 30\nBIG = (512, 512, 256)\n\ndef make_adam():\n    return keras.optimizers.Adam(learning_rate=0.001)\n\ndef make_early():\n    return keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)\n\nreg_configs = {\n    "Sin regularización":            dict(kw={},                          early=False),\n    "L2 (1e-4)":                     dict(kw=dict(l2=1e-4),               early=False),\n    "L1 (1e-5)":                     dict(kw=dict(l1=1e-5),               early=False),\n    "Dropout (0.3)":                 dict(kw=dict(dropout=0.3),           early=False),\n    "Early Stopping":                dict(kw={},                          early=True),\n    "L2 + Dropout + Early Stopping": dict(kw=dict(l2=1e-4, dropout=0.3),  early=True),\n}\n\nreg_histories, reg_models, reg_rows = {}, {}, []\nfor name, cfg in reg_configs.items():\n    keras.utils.set_random_seed(SEED)\n    model = build_model(make_adam(), units=BIG, **cfg["kw"])\n    cbs = [make_early()] if cfg["early"] else None\n    h = train(model, EPOCHS_REG, callbacks=cbs)\n\n    # Evaluación final con los pesos finales (con Early Stopping, los mejores restaurados)\n    _, acc_tr = model.evaluate(x_train, y_train, verbose=0)\n    _, acc_dv = model.evaluate(x_dev, y_dev, verbose=0)\n\n    reg_histories[name] = h\n    reg_models[name] = model\n    reg_rows.append({\n        "Configuración": name,\n        "Épocas": len(h.history["loss"]),\n        "Acc train": acc_tr,\n        "Acc dev": acc_dv,\n        "Error train (sesgo)": 1 - acc_tr,\n        "Error dev": 1 - acc_dv,\n        "Brecha (varianza)": acc_tr - acc_dv,\n    })\n    print(f"{name:32s} acc train = {acc_tr:.4f} | acc dev = {acc_dv:.4f}")\n\nreg_df = pd.DataFrame(reg_rows).round(4)\nreg_df.to_csv("resultados/regularizacion.csv", index=False)\ndisplay(reg_df)\n\n# Evidencia de overfitting: pérdida train vs dev del modelo sin regularización\nh0 = reg_histories["Sin regularización"]\nplt.figure(figsize=(8, 4.5))\nplt.plot(range(1, len(h0.history["loss"]) + 1), h0.history["loss"], label="Train")\nplt.plot(range(1, len(h0.history["val_loss"]) + 1), h0.history["val_loss"], label="Dev")\nplt.title("Modelo sin regularización: pérdida train vs dev")\nplt.xlabel("Época"); plt.ylabel("Pérdida"); plt.grid(alpha=0.3); plt.legend()\nplt.tight_layout()\nplt.savefig("figuras/05_overfitting_baseline.png", dpi=130)\nplt.show()\n\nplot_curves(reg_histories, "val_accuracy", "Accuracy en dev por técnica de regularización",\n            "Accuracy (dev)", "figuras/06_reg_val_accuracy.png")', 'metricas': 'best_name = reg_df.sort_values("Acc dev").iloc[-1]["Configuración"]\nbest_model = reg_models[best_name]\nprint("Modelo seleccionado (mejor en dev):", best_name)\n\ny_prob = best_model.predict(x_test, verbose=0)\ny_pred = y_prob.argmax(axis=1)\n\nfinal_metrics = {\n    "Accuracy":  accuracy_score(y_test, y_pred),\n    "Precision (macro)": precision_score(y_test, y_pred, average="macro"),\n    "Recall (macro)":    recall_score(y_test, y_pred, average="macro"),\n    "F1 (macro)":        f1_score(y_test, y_pred, average="macro"),\n}\nfor k, v in final_metrics.items():\n    print(f"{k:20s}: {v:.4f}")\n\nprint()\nprint(classification_report(y_test, y_pred, digits=4))\nreport_dict = classification_report(y_test, y_pred, digits=4, output_dict=True)\n\n# Matriz de confusión\ncm = confusion_matrix(y_test, y_pred)\nfig, ax = plt.subplots(figsize=(7, 6))\nConfusionMatrixDisplay(cm, display_labels=list(range(10))).plot(\n    ax=ax, cmap="Blues", values_format="d", colorbar=False)\nax.set_title(f"Matriz de confusión en test - {best_name}")\nplt.tight_layout()\nplt.savefig("figuras/07_matriz_confusion.png", dpi=130)\nplt.show()\n\n# Confusiones más frecuentes (fuera de la diagonal)\ncm_off = cm.copy()\nnp.fill_diagonal(cm_off, 0)\nidx = np.argsort(cm_off.ravel())[::-1][:3]\nconfusions = [(int(i // 10), int(i % 10), int(cm_off.ravel()[i])) for i in idx]\nfor real, pred, n in confusions:\n    print(f"Real {real} predicho como {pred}: {n} veces")\n\npd.DataFrame([final_metrics]).round(4).to_csv("resultados/metricas_test.csv", index=False)'}

PDF_NAME = "Informe_Redes_Neuronales_III.pdf"

ss = getSampleStyleSheet()
H1 = ParagraphStyle("H1", parent=ss["Heading1"], fontSize=15, spaceBefore=10, spaceAfter=6,
                    textColor=colors.HexColor("#1F3A5F"))
H2 = ParagraphStyle("H2", parent=ss["Heading2"], fontSize=12, spaceBefore=8, spaceAfter=4,
                    textColor=colors.HexColor("#1F3A5F"))
BODY = ParagraphStyle("BODY", parent=ss["BodyText"], fontSize=10, leading=14, alignment=TA_JUSTIFY,
                      spaceAfter=6)
CAP = ParagraphStyle("CAP", parent=ss["BodyText"], fontSize=8.5, leading=11, alignment=TA_CENTER,
                     textColor=colors.HexColor("#555555"), spaceAfter=8)
CODE_ST = ParagraphStyle("CODE", fontName="Courier", fontSize=6.8, leading=8.4,
                         backColor=colors.HexColor("#F3F3F3"), borderPadding=4, spaceAfter=8)
TITLE = ParagraphStyle("TITLE", parent=ss["Title"], fontSize=22, leading=28,
                       textColor=colors.HexColor("#1F3A5F"))
CENTER = ParagraphStyle("CENTER", parent=BODY, alignment=TA_CENTER, fontSize=11)

def pct(v):
    return f"{v * 100:.2f}%"

def P(text, style=BODY):
    return Paragraph(text, style)

def code_block(key):
    lines = []
    for ln in CODE[key].splitlines():
        lines.extend(textwrap.wrap(ln, 105, subsequent_indent="    ",
                                   replace_whitespace=False, drop_whitespace=False) or [""])
    return Preformatted("\n".join(lines), CODE_ST)

def fig(path, caption, width=5.8 * inch):
    iw, ih = ImageReader(path).getSize()
    return KeepTogether([Image(path, width=width, height=width * ih / iw), P(caption, CAP)])

def df_table(df, col_widths=None):
    data = [list(df.columns)]
    for row in df.itertuples(index=False):
        data.append([f"{c:.4f}" if isinstance(c, float) else str(c) for c in row])
    t = Table(data, colWidths=col_widths, repeatRows=1)
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1F3A5F")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTSIZE", (0, 0), (-1, -1), 7.5),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("ALIGN", (1, 0), (-1, -1), "CENTER"),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#999999")),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#EEF2F7")]),
    ]))
    return t

def footer(canvas, doc):
    canvas.saveState()
    canvas.setFont("Helvetica", 8)
    canvas.setFillColor(colors.HexColor("#777777"))
    canvas.drawString(0.8 * inch, 0.5 * inch, "R1-A3-S7 Redes Neuronales III - " + AUTOR)
    canvas.drawRightString(letter[0] - 0.8 * inch, 0.5 * inch, f"Página {doc.page}")
    canvas.restoreState()

# ---------- textos dinámicos con los resultados reales ----------
best_opt = opt_df.sort_values("Acc dev").iloc[-1]
worst_opt = opt_df.sort_values("Acc dev").iloc[0]
adam_rows = opt_df[opt_df["Optimizador"].str.startswith("Adam")]
adam_txt = "; ".join(f"{r['Optimizador']}: {pct(r['Acc dev'])}" for _, r in adam_rows.iterrows())

base = reg_df[reg_df["Configuración"] == "Sin regularización"].iloc[0]
best_reg = reg_df[reg_df["Configuración"] == best_name].iloc[0]
gap_base, gap_best = base["Brecha (varianza)"], best_reg["Brecha (varianza)"]
if gap_best < gap_base:
    reg_txt = (f"La brecha entre train y dev pasó de {gap_base * 100:.2f} puntos porcentuales en el modelo "
               f"sin regularización a {gap_best * 100:.2f} en la configuración seleccionada ({best_name}), "
               f"lo que indica una reducción de la varianza.")
else:
    reg_txt = (f"En este experimento la configuración seleccionada ({best_name}) presentó una brecha de "
               f"{gap_best * 100:.2f} puntos porcentuales frente a {gap_base * 100:.2f} del modelo sin "
               f"regularización; es decir, la regularización no redujo la brecha, algo posible en MNIST "
               f"porque es un problema relativamente simple en el que el sobreajuste es moderado.")
if best_reg["Acc dev"] > base["Acc dev"]:
    reg_txt += (f" Además, el accuracy en dev mejoró de {pct(base['Acc dev'])} a {pct(best_reg['Acc dev'])}.")
else:
    reg_txt += (f" El accuracy en dev fue {pct(best_reg['Acc dev'])} frente a {pct(base['Acc dev'])} del "
                f"modelo sin regularización.")

conf_txt = "; ".join(f"clase real {a} predicha como {b} ({n} veces)" for a, b, n in confusions)

story = []

# ---------- PORTADA ----------
story += [Spacer(1, 1.6 * inch),
          P("R1-A3-S7 Redes Neuronales III", TITLE),
          Spacer(1, 0.2 * inch),
          P("Optimización, regularización y evaluación de redes neuronales con TensorFlow + Keras", CENTER),
          Spacer(1, 0.5 * inch),
          P("<b>Curso:</b> Deep Learning - Conceptos (604013)", CENTER),
          P("<b>Programa:</b> Especialización en Inteligencia Artificial", CENTER),
          P(f"<b>Autor:</b> {AUTOR}", CENTER),
          P("<b>Fecha:</b> octubre de 2026", CENTER),
          Spacer(1, 0.4 * inch),
          P(f"<b>Repositorio:</b> {GITHUB_URL}", CENTER),
          PageBreak()]

# ---------- 1. INTRODUCCIÓN ----------
story += [P("1. Introducción y objetivo", H1),
          P("Este informe documenta la implementación, optimización y evaluación de una red neuronal "
            "multicapa (MLP) para clasificar dígitos manuscritos del dataset MNIST, usando TensorFlow + Keras "
            "en Google Colab. El objetivo es aplicar técnicas de optimización y regularización, analizar el "
            "compromiso entre sesgo y varianza, y evaluar el modelo con métricas de clasificación sobre un "
            "conjunto de prueba independiente."),
          P("Todo el código se encuentra en el notebook del repositorio de GitHub indicado en la portada y en la "
            "sección 9. Las tablas y gráficas de este documento fueron generadas por la propia ejecución del "
            "notebook."),
          ]

# ---------- 2. MARCO CONCEPTUAL ----------
story += [P("2. Marco conceptual", H1),
          P("<b>Hiperparámetros.</b> Son valores que se fijan antes del entrenamiento y que no se aprenden de los "
            "datos, a diferencia de los pesos y sesgos. Ejemplos: learning rate, tamaño de lote, número de "
            "épocas, cantidad de capas y neuronas, optimizador, tasa de Dropout y coeficientes L1/L2. Su "
            "elección se hace comparando el desempeño en el conjunto de validación (dev)."),
          P("<b>Conjuntos train / dev / test.</b> El conjunto de entrenamiento ajusta los pesos; el de "
            "validación (dev) permite comparar hiperparámetros y detectar sobreajuste; el de prueba se usa una "
            "única vez al final para estimar el desempeño en datos nuevos. Usar test para tomar decisiones "
            "contaminaría la estimación y la haría optimista."),
          P("<b>Optimización.</b> SGD actualiza los pesos con el gradiente de cada mini-batch. Momentum acumula "
            "una velocidad de gradientes pasados, suavizando las oscilaciones. RMSprop adapta el learning rate "
            "de cada parámetro según la magnitud reciente de sus gradientes. Adam combina momentum y "
            "adaptación del learning rate (Kingma & Ba, 2015). El learning rate es el hiperparámetro más "
            "sensible: si es muy alto el entrenamiento se vuelve inestable y si es muy bajo converge con "
            "lentitud."),
          P("<b>Regularización.</b> Conjunto de técnicas que reducen el sobreajuste (overfitting). L1 penaliza "
            "la suma de los valores absolutos de los pesos y favorece pesos nulos; L2 penaliza la suma de sus "
            "cuadrados y mantiene pesos pequeños; Dropout apaga aleatoriamente una fracción de neuronas durante "
            "el entrenamiento (Srivastava et al., 2014); Early Stopping detiene el entrenamiento cuando la "
            "pérdida en validación deja de mejorar."),
          P("<b>Bias, variance y tradeoff.</b> Un sesgo alto (bias) implica subajuste: el modelo no aprende "
            "bien ni siquiera los datos de entrenamiento, y se observa como un error alto en train. Una varianza "
            "alta implica sobreajuste: el modelo memoriza el entrenamiento y generaliza mal, y se observa como "
            "una brecha grande entre el error en dev y en train. El tradeoff consiste en encontrar la "
            "complejidad y regularización que reduzcan la varianza sin elevar en exceso el sesgo."),
          P("<b>Métricas de clasificación.</b> Accuracy es la proporción de aciertos. Precision es la fracción "
            "de predicciones de una clase que fueron correctas; Recall es la fracción de ejemplos reales de "
            "una clase que el modelo detectó; F1 es su media armónica. La matriz de confusión muestra, por "
            "clase real, cuántas veces se predijo cada clase. Para métricas globales se usó promedio macro, "
            "que da el mismo peso a cada clase."),
          P("<b>TensorFlow + Keras y PyTorch.</b> Keras es la API de alto nivel de TensorFlow: permite definir, "
            "compilar y entrenar modelos con pocas líneas (Sequential, compile, fit). PyTorch ofrece un estilo "
            "más imperativo, con un bucle de entrenamiento escrito explícitamente, lo que da más control. En "
            "este trabajo se utilizó TensorFlow + Keras."),
          ]

# ---------- 3. DATOS ----------
story += [P("3. Distribución y preprocesamiento de datos", H1),
          P(f"Se utilizó MNIST: imágenes en escala de grises de 28x28 píxeles con dígitos del 0 al 9. Los 60.000 "
            f"datos de entrenamiento originales se dividieron de forma estratificada en "
            f"<b>{n_train:,}</b> para train y <b>{n_dev:,}</b> para dev, y se conservaron los "
            f"<b>{n_test:,}</b> datos de prueba oficiales como test. Los píxeles se normalizaron de [0, 255] a "
            f"[0, 1] y cada imagen se aplanó a un vector de 784 características.".replace(",", ".")),
          code_block("datos"),
          fig("figuras/01_muestras_mnist.png", "Figura 1. Muestras del dataset MNIST.", 5.2 * inch),
          fig("figuras/02_distribucion_clases.png",
              "Figura 2. Distribución de clases en el conjunto de entrenamiento.", 4.6 * inch),
          ]

# ---------- 4. MODELO ----------
story += [P("4. Arquitectura y funciones de entrenamiento", H1),
          P("Se definió una función <b>build_model</b> que construye un MLP con capas densas de activación "
            "ReLU y una capa de salida softmax de 10 neuronas, entrenado con entropía cruzada categórica "
            "dispersa. La función recibe como parámetros los hiperparámetros a estudiar (optimizador, neuronas "
            "por capa, Dropout, L1 y L2), lo que permite experimentar cambiando una sola variable a la vez."),
          code_block("funciones"),
          ]

# ---------- 5. OPTIMIZACIÓN ----------
story += [P("5. Técnicas de optimización", H1),
          P("Se entrenó la misma arquitectura (784-256-128-10) durante 10 épocas, con batch de 128 y la misma "
            "semilla de inicialización, variando únicamente el optimizador y el learning rate."),
          code_block("optimizacion"),
          P("Resultados (accuracy y pérdida en la última época):", BODY),
          df_table(opt_df),
          Spacer(1, 8),
          fig("figuras/03_opt_val_loss.png", "Figura 3. Pérdida en el conjunto dev por optimizador."),
          fig("figuras/04_opt_val_accuracy.png", "Figura 4. Accuracy en el conjunto dev por optimizador."),
          P(f"<b>Análisis.</b> El mejor desempeño en dev lo obtuvo <b>{best_opt['Optimizador']}</b> con "
            f"{pct(best_opt['Acc dev'])}, mientras que el más bajo fue <b>{worst_opt['Optimizador']}</b> con "
            f"{pct(worst_opt['Acc dev'])}. Respecto al learning rate en Adam: {adam_txt}. Esto evidencia que "
            f"tanto el optimizador como el learning rate son hiperparámetros que condicionan la velocidad de "
            f"convergencia y la calidad final del modelo. Los optimizadores adaptativos (RMSprop y Adam) "
            f"suelen converger más rápido que SGD simple con el mismo número de épocas, y momentum acelera a "
            f"SGD al acumular la dirección de los gradientes."),
          ]

# ---------- 6. REGULARIZACIÓN ----------
story += [P("6. Regularización, bias y variance", H1),
          P("Para hacer observable el sobreajuste se utilizó una red más grande (784-512-512-256-10), entrenada "
            "hasta 30 épocas con Adam (lr = 0.001). Se compararon L2 (1e-4), L1 (1e-5), Dropout (0.3), Early "
            "Stopping (paciencia 3 con restauración de mejores pesos) y una combinación de L2, Dropout y Early "
            "Stopping."),
          code_block("regularizacion"),
          P("Resultados (el sesgo se aproxima con el error en train y la varianza con la brecha train - dev):", BODY),
          df_table(reg_df),
          Spacer(1, 8),
          fig("figuras/05_overfitting_baseline.png",
              "Figura 5. Pérdida en train y dev del modelo sin regularización."),
          fig("figuras/06_reg_val_accuracy.png",
              "Figura 6. Accuracy en dev según la técnica de regularización."),
          P(f"<b>Análisis.</b> {reg_txt} En la Figura 5 se aprecia la dinámica típica del sobreajuste: la "
            f"pérdida en train sigue bajando mientras que la de dev se estabiliza o crece. Early Stopping "
            f"actúa sobre este punto deteniendo el entrenamiento antes de memorizar el conjunto de "
            f"entrenamiento; L1 y L2 limitan la magnitud de los pesos; y Dropout obliga a la red a no depender "
            f"de neuronas individuales. El tradeoff se observa al comparar el error en train (sesgo) con la "
            f"brecha: regularizar en exceso puede aumentar el error en train sin mejorar la generalización."),
          ]

# ---------- 7. MÉTRICAS ----------
rep_rows = []
for c in range(10):
    r = report_dict[str(c)]
    rep_rows.append([str(c), f"{r['precision']:.4f}", f"{r['recall']:.4f}", f"{r['f1-score']:.4f}",
                     str(int(r['support']))])
rep_tbl = Table([["Clase", "Precision", "Recall", "F1", "Soporte"]] + rep_rows, repeatRows=1)
rep_tbl.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1F3A5F")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("FONTSIZE", (0, 0), (-1, -1), 8),
    ("ALIGN", (0, 0), (-1, -1), "CENTER"),
    ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#999999")),
    ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#EEF2F7")]),
]))
glob_df = pd.DataFrame([{k: round(v, 4) for k, v in final_metrics.items()}])

story += [P("7. Evaluación final con métricas", H1),
          P(f"Con base en el accuracy en dev se seleccionó la configuración <b>{best_name}</b> y se evaluó una "
            f"única vez sobre los {n_test:,} datos de test.".replace(",", ".")),
          code_block("metricas"),
          P("Métricas globales (promedio macro):", BODY),
          df_table(glob_df),
          Spacer(1, 8),
          P("Métricas por clase:", BODY),
          rep_tbl,
          Spacer(1, 8),
          fig("figuras/07_matriz_confusion.png", "Figura 7. Matriz de confusión en el conjunto de test.", 4.8 * inch),
          P(f"<b>Análisis.</b> El modelo alcanzó un accuracy de {pct(final_metrics['Accuracy'])} en test, con "
            f"F1 macro de {pct(final_metrics['F1 (macro)'])}. Las confusiones más frecuentes fueron: "
            f"{conf_txt}. Estas parejas corresponden a dígitos con trazos visualmente similares. Como el "
            f"conjunto de test no se usó para elegir hiperparámetros, estos valores son una estimación "
            f"honesta del desempeño en datos nuevos."),
          ]

# ---------- 8. CONCLUSIONES ----------
story += [P("8. Conclusiones", H1),
          P(f"1) La separación en train, dev y test permitió comparar hiperparámetros sin contaminar la "
            f"evaluación final. 2) El optimizador y el learning rate influyeron de forma clara en la "
            f"convergencia: el mejor optimizador en dev fue {best_opt['Optimizador']} "
            f"({pct(best_opt['Acc dev'])}). 3) {reg_txt} 4) El modelo final ({best_name}) obtuvo "
            f"{pct(final_metrics['Accuracy'])} de accuracy en test, con precision macro de "
            f"{pct(final_metrics['Precision (macro)'])} y recall macro de {pct(final_metrics['Recall (macro)'])}. "
            f"5) La matriz de confusión permitió identificar qué dígitos se confunden más, información que no "
            f"revela el accuracy global. Como trabajo futuro se podrían explorar redes convolucionales, "
            f"búsqueda sistemática de hiperparámetros y decaimiento del learning rate."),
          ]

# ---------- 9. REPOSITORIO ----------
story += [P("9. Repositorio y reproducibilidad", H1),
          P(f"Notebook, figuras y resultados (CSV) disponibles en: <b>{GITHUB_URL}</b>"),
          P("Para reproducir: abrir el notebook en Google Colab y ejecutar todas las celdas. Se fijó la semilla "
            "42; pueden existir pequeñas variaciones numéricas según la versión de TensorFlow y el hardware."),
          P("Referencias", H2),
          P("Chollet, F. (2021). <i>Deep Learning with Python</i> (2.a ed.). Manning."),
          P("Goodfellow, I., Bengio, Y., &amp; Courville, A. (2016). <i>Deep Learning</i>. MIT Press."),
          P("Kingma, D. P., &amp; Ba, J. (2015). Adam: A method for stochastic optimization. "
            "<i>3rd International Conference on Learning Representations (ICLR)</i>."),
          P("LeCun, Y., Cortes, C., &amp; Burges, C. J. C. (1998). <i>The MNIST database of handwritten digits</i>."),
          P("Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I., &amp; Salakhutdinov, R. (2014). Dropout: "
            "A simple way to prevent neural networks from overfitting. <i>Journal of Machine Learning "
            "Research, 15</i>, 1929-1958."),
          ]

doc = SimpleDocTemplate(PDF_NAME, pagesize=letter, leftMargin=0.8 * inch, rightMargin=0.8 * inch,
                        topMargin=0.8 * inch, bottomMargin=0.8 * inch,
                        title="R1-A3-S7 Redes Neuronales III", author=AUTOR)
doc.build(story, onFirstPage=lambda c, d: None, onLaterPages=footer)
print("PDF generado:", PDF_NAME)
if "TU_USUARIO" in GITHUB_URL:
    print("ATENCIÓN: cambia GITHUB_URL por el enlace real de tu repositorio y vuelve a ejecutar esta celda.")

try:
    from google.colab import files
    files.download(PDF_NAME)
except Exception:
    pass